In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate

dataset = load_dataset("eriktks/conll2003")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True
    )
    word_ids = tokenized.word_ids()
    labels = example["pos_tags"]

    aligned_labels = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
        elif word_id != prev_word_id:
            aligned_labels.append(labels[word_id])
        else:
            aligned_labels.append(-100)
        prev_word_id = word_id
    tokenized["labels"] = aligned_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)

num_labels = len(dataset["train"].features["pos_tags"].feature.names)
id2label = {i: label for i, label in enumerate(dataset["train"].features["pos_tags"].feature.names)}
label2id = {label: i for i, label in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50
)



Using the latest cached version of the dataset since eriktks/conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\saksh\.cache\huggingface\datasets\eriktks___conll2003\default\0.0.0\ce85b39f9dd99f552d0739d456814e95fb6a39b0 (last modified on Mon Apr  6 20:46:17 2026).


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [ ]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = []
    true_labels = []
    for pred_seq, label_seq in zip(predictions, labels):
        temp_preds = []
        temp_labels = []
        for p_i, l_i in zip(pred_seq, label_seq):
            if l_i != -100:
                temp_preds.append(id2label[p_i])
                temp_labels.append(id2label[l_i])
        true_predictions.append(temp_preds)
        true_labels.append(temp_labels)
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("./my_model")

eval_results = trainer.evaluate()
print(eval_results)

sentence = "John works at Google in California"
inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
predicted_labels = [id2label[p.item()] for p in predictions[0]]

for token, label in zip(tokens, predicted_labels):
    if not token.startswith("##"):
        print(f"{token:12} → {label}")

In [ ]:
np.argmax(predictions, axis=2)
if l_i != -100:
    id2label[p_i]
metric.compute(...)

In [ ]:
trainer.evaluate()

In [ ]:
model = AutoModelForTokenClassification.from_pretrained("./my_model")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
sentence = "John works at Google in California"
inputs = tokenizer(
    sentence,
    return_tensors="pt",
    truncation=True
)
outputs = model(**inputs)
logits = outputs.logits

In [ ]:

predictions = torch.argmax(logits, dim=2)
predicted_labels = [
    id2label[p.item()] for p in predictions[0]
]
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
for token, label in zip(tokens, predicted_labels):
    print(f"{token:12} → {label}")